In [1]:
# Enable auto-reload for imported modules
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Get project root (from training/notebooks/ go up 2 levels)
project_root = Path.cwd().parent.parent  

# Add paths
sys.path.insert(0, str(project_root))

# Verify paths
print("✓ Paths added to sys.path")

from training import settings
import os

os.environ["TRANSFORMERS_CACHE"] = str(settings.TRANSFORMER_CACHE_DIR)
os.environ["HF_HOME"] = str(settings.TRANSFORMER_DATASETS_DIR)

✓ Paths added to sys.path


In [1]:
from datasets import Dataset, load_dataset

persona_ds = load_dataset("proj-persona/PersonaHub", "persona")


/Users/maroon/.pyenv/versions/3.12.7/envs/tiny-model/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Randomly get 20 personas
import random

sampled_personas = persona_ds['train'].shuffle(seed=40).select(range(20))
personas = [p['persona'] for p in sampled_personas]
personas_txt = "\n- ".join(personas)

print("✓ Persona dataset loaded and sampled", personas_txt)


✓ Persona dataset loaded and sampled A Political Analyst specialized in El Salvador's political landscape.
- A legal advisor who understands the legal implications of incomplete or inaccurate project documentation
- A maternal health advocate focused on raising awareness about postpartum complications.
- A school basketball team captain who believes sports and their funding should be prioritized over student council campaigns
- A determined basketball player who aspires to be the star athlete of the school
- A virtual reality content creator sharing their experiences and creations on a popular online platform
- An engineer with a shared sense of humor, who has known the comedian since grade school
- an IT project manager who adopted extreme programming (XP) methodologies on his own team.
- a newly hired general counsel at TurpCo Industries
- A divorced father of three seeking legal representation for child custody matters
- a geography teacher who was born and raised in Antigua and Bar

In [ ]:
from src.payment_classifier.llm import LiteLLMProvider, LLMSettings
from training.config.settings import settings

llm = LiteLLMProvider(LLMSettings(
    api_key=settings.OPENAI_API_KEY,
    llm_model_name="gpt-5o-mini"
))

In [ ]:
from pydantic import BaseModel
from typing import List

# Prompts for generating `PAYMENT_INTENTION` values
prompt = '''You are generating training data for a payment intention detection model.
Create diverse payment-related chat messages with the following personas:
{persona}

## Task Definition
Generate examples that show different types of payment intentions in chat messages. The model needs to distinguish between:

- **PAYMENT_REQUEST**: User is asking/requesting someone to send them money, Ex: invoice, pay back, lend money, etc.
- **PAYMENT_SEND**: User intends to send/pay money to someone, Ex: I wanna send him $20, I want to pay the bill, etc.
- **PAYMENT_COMMAND**: User is instructing a system/app to make a payment
- **NO_PAYMENT**: No payment intention present (but may mention payment related things in other contexts)

## Requirements
1. **Payment Amounts**: 
   - Specific: $50, €25, 0.01 BTC, 1000 sats
   - General: "some money", "a few bucks", "the amount we discussed"
   - No amount: "pay me", "send money"

2. **Payment Methods**:
   - Digital: Venmo, PayPal, CashApp, Zelle, Apple Pay, Google Pay
   - Crypto: Bitcoin, Ethereum, USDC, Lightning Network
   - Traditional: cash, check, bank transfer, wire
   - Generic: "pay", "send money", "transfer"
    - No method mentioned

3. **Language Styles**:
   - Formal: "Please transfer the payment for the invoice"
   - Casual: "Can you Venmo me for dinner?"
   - Slang: "Spot me twenty", "Hit me up with that cash"
   - Commands: "Send $50 to John", "Pay the electricity bill"

## Formatting Requirements
    - Ensure no bigram reuse per generated example. 
    - Each example should be unique and not repeat phrases from other examples.
    - Possible include some msats, malformed invoices, fee caps, route hints.
    - Ranging 20-30% typos/formal variants.
'''

class Samples(BaseModel):
    samples: List[str]

personas = get_random_persona(20)
personas_str = "\n".join([f"- {p}" for p in personas])

outputs = llm.generate_structured_output([
    {"role": "system", "content": prompt.format(persona=personas_str)},
], Samples)


RuntimeError: asyncio.run() cannot be called from a running event loop